# General Tasks 1

Importing files [Komponente_K7.csv](data/Logistikverzug/Komponente_K7.csv) and [Logistikverzug_K7.csv](data/Logistikverzug/Logistikverzug_K7.csv) and taking a look at the data and filetypes 


In [12]:
import pandas as pd
from IPython.display import display

# Import Data
komponente_k7 = pd.read_csv("data/Logistikverzug/Komponente_K7.csv", sep=";", index_col=0)
logistikverzug_k7 = pd.read_csv("data/Logistikverzug/Logistikverzug_K7.csv", sep=",", index_col=0)

# Display Data
display(komponente_k7.sort_values(by='IDNummer').reset_index(drop=True).head())
display(logistikverzug_k7.sort_values(by='IDNummer').reset_index(drop=True).head())


,IDNummer,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft
0,K7-113-1132-1,2012-11-12,112,1132,0
1,K7-113-1132-10,2012-11-12,112,1132,0
2,K7-113-1132-100,2012-11-13,112,1132,0
3,K7-113-1132-1000,2012-11-22,112,1132,0
4,K7-113-1132-10000,2013-02-15,112,1132,0


,IDNummer,Wareneingang,Herstellernummer,Werksnummer,Fehlerhaft
0,K7-113-1132-1,2012-11-18,112,1132,0
1,K7-113-1132-10,2012-11-19,112,1132,0
2,K7-113-1132-100,2012-11-18,112,1132,0
3,K7-113-1132-1000,2012-11-30,112,1132,0
4,K7-113-1132-10000,2013-02-22,112,1132,0


* Creating a new dataset 'Logistics delay' by merging the two datasets on the 'IDNummer' column.
* Assuming that produced goods are issued one day after the production date

In [13]:
import numpy as np

# Merge DataFrames
logistics_delay = pd.merge(komponente_k7[['IDNummer', 'Produktionsdatum']], logistikverzug_k7[['IDNummer', 'Wareneingang']], on='IDNummer', how='inner')

# Converting to datetime datatypes
logistics_delay['Produktionsdatum'] = pd.to_datetime(logistics_delay['Produktionsdatum'])
logistics_delay['Wareneingang'] = pd.to_datetime(logistics_delay['Wareneingang'])
display(logistics_delay.dtypes)

# Calculating Ausgabedatum and Verzug column needed for later analysis
logistics_delay['Ausgabedatum'] = logistics_delay['Produktionsdatum'] + pd.Timedelta(days=1)
logistics_delay['Verzug'] = (logistics_delay['Wareneingang'] - logistics_delay['Ausgabedatum']).dt.days
logistics_delay['Verzug_AT'] = np.busday_count(logistics_delay['Ausgabedatum'].values.astype('datetime64[D]'), logistics_delay['Wareneingang'].values.astype('datetime64[D]')) # type: ignore

# Display stats
display(logistics_delay.head())
display(logistics_delay['Verzug'].describe())


IDNummer                       str
Produktionsdatum    datetime64[us]
Wareneingang        datetime64[us]
dtype: object

,IDNummer,Produktionsdatum,Wareneingang,Ausgabedatum,Verzug,Verzug_AT
0,K7-114-1142-1,2008-11-12,2008-11-19,2008-11-13,6,4
1,K7-114-1142-2,2008-11-12,2008-11-19,2008-11-13,6,4
2,K7-114-1142-3,2008-11-13,2008-11-20,2008-11-14,6,4
3,K7-114-1142-4,2008-11-13,2008-11-20,2008-11-14,6,4
4,K7-114-1142-5,2008-11-13,2008-11-19,2008-11-14,5,3


count    306490.000000
mean          6.080437
std           1.012302
min           3.000000
25%           5.000000
50%           6.000000
75%           7.000000
max          14.000000
Name: Verzug, dtype: float64

## b) Interpreting the logistics delay and discussing possible alternatives

* The working day delay is reduced, due to removing the weekends. That would reduce the mean delay time considerably because the mean was at around 6 days before.
* If the warehouse does work on weekends then it would not make sense to remove the weekends
* This calculations does not include german holidays which could be important to assess aswell depending on the warehouse operation days.
* the generalized issue date does shift all dates by one day. Including 'Ausgabedatum' weekdays into weekends and vice versa. Important to be aware of that.